In [1]:
import cv2
import json
import numpy as np
import os

IMAGE_FOLDER = "test_images/images"
OUTPUT_FOLDER = "output_json"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

image_files = sorted([
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

def line_equation(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    if x2 - x1 == 0:
        return None, None, f"x = {x1}"
    m = (y2 - y1) / (x2 - x1)
    c = y1 - m * x1
    return m, c, f"y = {m:.2f}x + {c:.2f}"

def draw_point_label(img, pt, label, width, height):
    cx, cy = int(pt[0]), int(pt[1])
    if 0 <= cx < width and 0 <= cy < height:
        cv2.circle(img, (cx, height - cy), 5, (0, 0, 0), -1)
        cv2.putText(img, label, (cx + 8, height - cy - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

def draw_shaded_region(frame, A, B, C):
    height, width = frame.shape[:2]
    overlay = np.zeros_like(frame)
    m_ab, c_ab, eq_ab = line_equation(A, B)
    m_bc, c_bc, eq_bc = line_equation(B, C)

    result = {
        "A": A, "B": B, "C": C,
        "line_AB": eq_ab, "line_BC": eq_bc
    }

    if m_ab is not None:
        Q = (-c_ab / m_ab, 0)
        P = (0, c_ab)
        O = (0, 0)
        triangle = np.array([
            (int(Q[0]), height - int(Q[1])),
            (int(P[0]), height - int(P[1])),
            (int(O[0]), height - int(O[1]))
        ], dtype=np.int32)
        cv2.fillPoly(overlay, [triangle], (255, 0, 255))
        for pt, label in zip([Q, P, O], ['Q', 'P', 'O']):
            draw_point_label(overlay, pt, label, width, height)
        result.update({"Q": Q, "P": P, "O": O})
    else:
        result.update({"Q": None, "P": None, "O": None})

    if m_bc is not None:
        S = (0, c_bc)
        R = (width, m_bc * width + c_bc)
    else:
        S = (B[0], 0)
        R = (B[0], height)

    bottom_left = (0, 0)
    bottom_right = (width, 0)
    extended_area = np.array([
        (int(S[0]), height - int(S[1])),
        (int(R[0]), height - int(R[1])),
        (bottom_right[0], height - bottom_right[1]),
        (bottom_left[0], height - bottom_left[1])
    ], dtype=np.int32)
    cv2.fillPoly(overlay, [extended_area], (0, 200, 200))

    for pt, label in zip([A, B, C], ['A', 'B', 'C']):
        draw_point_label(overlay, pt, label, width, height)

    draw_point_label(overlay, S, "S", width, height)
    draw_point_label(overlay, R, "R", width, height)
    draw_point_label(overlay, bottom_left, "Bottom Left", width, height)
    draw_point_label(overlay, bottom_right, "Bottom Right", width, height)

    result.update({
        "S": S, "R": R,
        "bottom_left": bottom_left,
        "bottom_right": bottom_right
    })

    return overlay, result

for image_file in image_files:
    image_path = os.path.join(IMAGE_FOLDER, image_file)
    image_name = os.path.splitext(image_file)[0]
    json_path = os.path.join(OUTPUT_FOLDER, f"{image_name}_lshape.json")

    frame = cv2.imread(image_path)
    if frame is None:
        print(f"❌ Could not load {image_path}")
        continue

    height, width = frame.shape[:2]
    points = []
    overlay_drawn = None

    def click_callback(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
            math_y = height - y
            points.append((x, math_y))

    cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
    cv2.setMouseCallback("Define L-Shape ROI", click_callback)

    while True:
        display_frame = frame.copy()
        for (x, my) in points:
            cv2.circle(display_frame, (x, height - my), 7, (0, 0, 255), -1)

        if len(points) >= 2:
            cv2.line(display_frame, (points[0][0], height - points[0][1]),
                     (points[1][0], height - points[1][1]), (0, 255, 0), 2)
        if len(points) == 3:
            cv2.line(display_frame, (points[1][0], height - points[1][1]),
                     (points[2][0], height - points[2][1]), (0, 255, 0), 2)

        if overlay_drawn is not None:
            display_frame = cv2.addWeighted(display_frame, 1.0, overlay_drawn, 0.6, 0)

        cv2.putText(display_frame, f"{image_file} - Click 3 points: 's'=Save, 'q'=Skip, 'r'=Reset",
                    (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        cv2.imshow("Define L-Shape ROI", display_frame)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('r'):
            points = []
            overlay_drawn = None

        elif key == ord('s') and len(points) == 3:
            A, B, C = points
            overlay_drawn, result_data = draw_shaded_region(frame, A, B, C)
            with open(json_path, 'w') as f:
                json.dump(result_data, f, indent=2)
            print(f"[✅] Saved {json_path}")
            break

        elif key == ord('q'):
            print(f"[⚠] Skipped {image_file}")
            break

    cv2.destroyAllWindows()

print("\n✅ Batch L-shape definition complete.")



[✅] Saved output_json\a_lshape.json
[✅] Saved output_json\b_lshape.json
[✅] Saved output_json\c_lshape.json
[✅] Saved output_json\d_lshape.json

✅ Batch L-shape definition complete.


In [ ]:
import cv2
import json
import numpy as np
import os

IMAGE_FOLDER = "test_images/images"
OUTPUT_FOLDER = "output_json"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

image_files = sorted([
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

def line_equation(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    if x2 - x1 == 0:
        return None, None, f"x = {x1}"
    m = (y2 - y1) / (x2 - x1)
    c = y1 - m * x1
    return m, c, f"y = {m:.2f}x + {c:.2f}"

def draw_point_label(img, pt, label, width, height):
    """Draw a small dot and label at `pt`, which is expressed in math coords (origin bottom‑left)."""
    cx, cy = int(pt[0]), int(pt[1])
    if 0 <= cx < width and 0 <= cy < height:
        cv2.circle(img, (cx, height - cy), 5, (0, 0, 0), -1)
        cv2.putText(img, label, (cx + 8, height - cy - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

def draw_axes(img, width, height, step: int = 100):
    """Draw X‑ and Y‑axes with tick marks every `step` pixels (origin at bottom‑left)."""
    # Main axes
    cv2.line(img, (0, height), (0, 0), (200, 200, 200), 1)  # Y‑axis
    cv2.line(img, (0, height), (width, height), (200, 200, 200), 1)  # X‑axis

    # Arrow heads
    cv2.arrowedLine(img, (0, height), (0, 20), (200, 200, 200), 1, tipLength=0.025)
    cv2.arrowedLine(img, (0, height), (width - 20, height), (200, 200, 200), 1, tipLength=0.025)

    # Labels
    cv2.putText(img, "Y", (5, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    cv2.putText(img, "X", (width - 25, height - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

    # Tick marks and numbers on Y‑axis
    for y in range(step, height, step):
        cv2.line(img, (0, height - y), (7, height - y), (200, 200, 200), 1)
        cv2.putText(img, str(y), (10, height - y + 4),
                    cv2.FONT_HERSHEY_PLAIN, 0.7, (200, 200, 200), 1)

    # Tick marks and numbers on X‑axis
    for x in range(step, width, step):
        cv2.line(img, (x, height), (x, height - 7), (200, 200, 200), 1)
        cv2.putText(img, str(x), (x - 10, height - 10),
                    cv2.FONT_HERSHEY_PLAIN, 0.7, (200, 200, 200), 1)

def draw_shaded_region(frame, A, B, C):
    height, width = frame.shape[:2]
    overlay = np.zeros_like(frame)
    m_ab, c_ab, eq_ab = line_equation(A, B)
    m_bc, c_bc, eq_bc = line_equation(B, C)

    result = {
        "A": A, "B": B, "C": C,
        "line_AB": eq_ab, "line_BC": eq_bc
    }

    # --- Triangle under AB ---
    if m_ab is not None:
        Q = (-c_ab / m_ab, 0)
        P = (0, c_ab)
        O = (0, 0)
        triangle = np.array([
            (int(Q[0]), height - int(Q[1])),
            (int(P[0]), height - int(P[1])),
            (int(O[0]), height - int(O[1]))
        ], dtype=np.int32)
        cv2.fillPoly(overlay, [triangle], (255, 0, 255))
        for pt, label in zip([Q, P, O], ['Q', 'P', 'O']):
            draw_point_label(overlay, pt, label, width, height)
        result.update({"Q": Q, "P": P, "O": O})
    else:
        result.update({"Q": None, "P": None, "O": None})

    # --- Extended BC area ---
    if m_bc is not None:
        S = (0, c_bc)
        R = (width, m_bc * width + c_bc)
    else:
        S = (B[0], 0)
        R = (B[0], height)

    bottom_left = (0, 0)
    bottom_right = (width, 0)
    extended_area = np.array([
        (int(S[0]), height - int(S[1])),
        (int(R[0]), height - int(R[1])),
        (bottom_right[0], height - bottom_right[1]),
        (bottom_left[0], height - bottom_left[1])
    ], dtype=np.int32)
    cv2.fillPoly(overlay, [extended_area], (0, 200, 200))

    # --- Labels ---
    for pt, label in zip([A, B, C], ['A', 'B', 'C']):
        draw_point_label(overlay, pt, label, width, height)

    draw_point_label(overlay, S, "S", width, height)
    draw_point_label(overlay, R, "R", width, height)
    draw_point_label(overlay, bottom_left, "Bottom Left", width, height)
    draw_point_label(overlay, bottom_right, "Bottom Right", width, height)

    result.update({
        "S": S, "R": R,
        "bottom_left": bottom_left,
        "bottom_right": bottom_right
    })

    return overlay, result

# =====================================================================
# Main batch loop
# =====================================================================

for image_file in image_files:
    image_path = os.path.join(IMAGE_FOLDER, image_file)
    image_name = os.path.splitext(image_file)[0]
    json_path = os.path.join(OUTPUT_FOLDER, f"{image_name}_lshape.json")

    frame = cv2.imread(image_path)
    if frame is None:
        print(f"❌ Could not load {image_path}")
        continue

    height, width = frame.shape[:2]
    points = []
    overlay_drawn = None

    def click_callback(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
            math_y = height - y  # convert to math coords
            points.append((x, math_y))

    cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
    cv2.setMouseCallback("Define L-Shape ROI", click_callback)

    while True:
        display_frame = frame.copy()

        # NEW: draw axes first
        draw_axes(display_frame, width, height)

        # existing drawn points
        for (x, my) in points:
            cv2.circle(display_frame, (x, height - my), 7, (0, 0, 255), -1)

        # AB and BC lines if available
        if len(points) >= 2:
            cv2.line(display_frame, (points[0][0], height - points[0][1]),
                     (points[1][0], height - points[1][1]), (0, 255, 0), 2)
        if len(points) == 3:
            cv2.line(display_frame, (points[1][0], height - points[1][1]),
                     (points[2][0], height - points[2][1]), (0, 255, 0), 2)

        # overlay if already computed
        if overlay_drawn is not None:
            display_frame = cv2.addWeighted(display_frame, 1.0, overlay_drawn, 0.6, 0)

        cv2.putText(display_frame, f"{image_file} - Click 3 points: 's'=Save, 'q'=Skip, 'r'=Reset",
                    (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        cv2.imshow("Define L-Shape ROI", display_frame)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('r'):
            points = []
            overlay_drawn = None

        elif key == ord('s') and len(points) == 3:
            A, B, C = points
            overlay_drawn, result_data = draw_shaded_region(frame, A, B, C)
            with open(json_path, 'w') as f:
                json.dump(result_data, f, indent=2)
            print(f"[✅] Saved {json_path}")
            break

        elif key == ord('q'):
            print(f"[⚠] Skipped {image_file}")
            break

    cv2.destroyAllWindows()

print("\n✅ Batch L-shape definition complete.")


[✅] Saved output_json/a_lshape.json
[✅] Saved output_json/b_lshape.json
[✅] Saved output_json/c_lshape.json
[✅] Saved output_json/d_lshape.json
[✅] Saved output_json/plain_lshape.json

✅ Batch L-shape definition complete.


: 

In [ ]:
import cv2
import json
import numpy as np
import os

IMAGE_FOLDER = "test_images/images"
OUTPUT_FOLDER = "output_json"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

image_files = sorted([
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

def line_equation(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    if x2 - x1 == 0:
        return None, None, f"x = {x1}"
    m = (y2 - y1) / (x2 - x1)
    c = y1 - m * x1
    return m, c, f"y = {m:.2f}x + {c:.2f}"

def draw_point_label(img, pt, label, width, height):
    """Draw a small dot and label at `pt`, which is expressed in math coords (origin bottom‑left)."""
    cx, cy = int(pt[0]), int(pt[1])
    if 0 <= cx < width and 0 <= cy < height:
        cv2.circle(img, (cx, height - cy), 5, (0, 0, 0), -1)
        cv2.putText(img, label, (cx + 8, height - cy - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

def draw_axes(img, width, height, step: int = 100):
    """Draw X‑ and Y‑axes with tick marks every `step` pixels (origin at bottom‑left)."""
    # Main axes
    cv2.line(img, (0, height), (0, 0), (200, 200, 200), 1)  # Y‑axis
    cv2.line(img, (0, height), (width, height), (200, 200, 200), 1)  # X‑axis

    # Arrow heads
    cv2.arrowedLine(img, (0, height), (0, 20), (200, 200, 200), 1, tipLength=0.025)
    cv2.arrowedLine(img, (0, height), (width - 20, height), (200, 200, 200), 1, tipLength=0.025)

    # Labels
    cv2.putText(img, "Y", (5, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    cv2.putText(img, "X", (width - 25, height - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

    # Tick marks and numbers on Y‑axis
    for y in range(step, height, step):
        cv2.line(img, (0, height - y), (7, height - y), (200, 200, 200), 1)
        cv2.putText(img, str(y), (10, height - y + 4),
                    cv2.FONT_HERSHEY_PLAIN, 0.7, (200, 200, 200), 1)

    # Tick marks and numbers on X‑axis
    for x in range(step, width, step):
        cv2.line(img, (x, height), (x, height - 7), (200, 200, 200), 1)
        cv2.putText(img, str(x), (x - 10, height - 10),
                    cv2.FONT_HERSHEY_PLAIN, 0.7, (200, 200, 200), 1)

def draw_shaded_region(frame, A, B, C):
    height, width = frame.shape[:2]
    overlay = np.zeros_like(frame)
    m_ab, c_ab, eq_ab = line_equation(A, B)
    m_bc, c_bc, eq_bc = line_equation(B, C)

    result = {
        "A": A, "B": B, "C": C,
        "line_AB": eq_ab, "line_BC": eq_bc
    }

    # ============ EXTENDED LINE AB - Find intersection points ============
    if m_ab is not None:
        # AB line intersections with image boundaries
        ab_left = (0, c_ab)  # intersection with left edge (x=0)
        ab_right = (width, m_ab * width + c_ab)  # intersection with right edge (x=width)
        ab_bottom = (-c_ab / m_ab, 0)  # intersection with bottom edge (y=0)
        ab_top = ((height - c_ab) / m_ab, height)  # intersection with top edge (y=height)

        # Area 1: BELOW extended line AB (Magenta)
        area1_below_ab = np.array([
            (int(ab_left[0]), height - int(ab_left[1])),
            (int(ab_right[0]), height - int(ab_right[1])),
            (width, height),  # bottom-right corner
            (0, height)  # bottom-left corner
        ], dtype=np.int32)
        cv2.fillPoly(overlay, [area1_below_ab], (255, 0, 255))

        # Area 2: ABOVE extended line AB (Yellow)
        area2_above_ab = np.array([
            (int(ab_left[0]), height - int(ab_left[1])),
            (int(ab_right[0]), height - int(ab_right[1])),
            (width, 0),  # top-right corner
            (0, 0)  # top-left corner
        ], dtype=np.int32)
        cv2.fillPoly(overlay, [area2_above_ab], (0, 255, 255))

        result.update({
            "ab_left": ab_left,
            "ab_right": ab_right,
            "ab_bottom": ab_bottom,
            "ab_top": ab_top
        })
    else:
        # Vertical line AB (x = constant)
        x_ab = A[0]
        ab_left = None
        ab_right = None
        ab_bottom = (x_ab, 0)
        ab_top = (x_ab, height)

        # Area 1: LEFT of vertical line AB (Magenta)
        area1_left_ab = np.array([
            (int(x_ab), 0),
            (int(x_ab), height),
            (0, height),
            (0, 0)
        ], dtype=np.int32)
        cv2.fillPoly(overlay, [area1_left_ab], (255, 0, 255))

        # Area 2: RIGHT of vertical line AB (Yellow)
        area2_right_ab = np.array([
            (int(x_ab), 0),
            (int(x_ab), height),
            (width, height),
            (width, 0)
        ], dtype=np.int32)
        cv2.fillPoly(overlay, [area2_right_ab], (0, 255, 255))

        result.update({
            "ab_left": ab_left,
            "ab_right": ab_right,
            "ab_bottom": ab_bottom,
            "ab_top": ab_top
        })

    # ============ EXTENDED LINE BC - Find intersection points ============
    if m_bc is not None:
        # BC line intersections with image boundaries
        bc_left = (0, c_bc)  # intersection with left edge (x=0)
        bc_right = (width, m_bc * width + c_bc)  # intersection with right edge (x=width)
        bc_bottom = (-c_bc / m_bc, 0)  # intersection with bottom edge (y=0)
        bc_top = ((height - c_bc) / m_bc, height)  # intersection with top edge (y=height)

        # Area 3: BELOW extended line BC (Cyan)
        area3_below_bc = np.array([
            (int(bc_left[0]), height - int(bc_left[1])),
            (int(bc_right[0]), height - int(bc_right[1])),
            (width, height),  # bottom-right corner
            (0, height)  # bottom-left corner
        ], dtype=np.int32)
        cv2.fillPoly(overlay, [area3_below_bc], (255, 255, 0))

        # Area 4: ABOVE extended line BC (Green)
        area4_above_bc = np.array([
            (int(bc_left[0]), height - int(bc_left[1])),
            (int(bc_right[0]), height - int(bc_right[1])),
            (width, 0),  # top-right corner
            (0, 0)  # top-left corner
        ], dtype=np.int32)
        cv2.fillPoly(overlay, [area4_above_bc], (0, 255, 0))

        result.update({
            "bc_left": bc_left,
            "bc_right": bc_right,
            "bc_bottom": bc_bottom,
            "bc_top": bc_top
        })
    else:
        # Vertical line BC (x = constant)
        x_bc = B[0]
        bc_left = None
        bc_right = None
        bc_bottom = (x_bc, 0)
        bc_top = (x_bc, height)

        # Area 3: LEFT of vertical line BC (Cyan)
        area3_left_bc = np.array([
            (int(x_bc), 0),
            (int(x_bc), height),
            (0, height),
            (0, 0)
        ], dtype=np.int32)
        cv2.fillPoly(overlay, [area3_left_bc], (255, 255, 0))

        # Area 4: RIGHT of vertical line BC (Green)
        area4_right_bc = np.array([
            (int(x_bc), 0),
            (int(x_bc), height),
            (width, height),
            (width, 0)
        ], dtype=np.int32)
        cv2.fillPoly(overlay, [area4_right_bc], (0, 255, 0))

        result.update({
            "bc_left": bc_left,
            "bc_right": bc_right,
            "bc_bottom": bc_bottom,
            "bc_top": bc_top
        })

    # --- Labels for clicked points ---
    for pt, label in zip([A, B, C], ['A', 'B', 'C']):
        draw_point_label(overlay, pt, label, width, height)

    return overlay, result

# =====================================================================
# Main batch loop
# =====================================================================

for image_file in image_files:
    image_path = os.path.join(IMAGE_FOLDER, image_file)
    image_name = os.path.splitext(image_file)[0]
    json_path = os.path.join(OUTPUT_FOLDER, f"{image_name}_lshape.json")

    frame = cv2.imread(image_path)
    if frame is None:
        print(f"❌ Could not load {image_path}")
        continue

    height, width = frame.shape[:2]
    points = []
    overlay_drawn = None

    def click_callback(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
            math_y = height - y  # convert to math coords
            points.append((x, math_y))

    cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
    cv2.setMouseCallback("Define L-Shape ROI", click_callback)

    while True:
        display_frame = frame.copy()

        # NEW: draw axes first
        draw_axes(display_frame, width, height)

        # existing drawn points
        for (x, my) in points:
            cv2.circle(display_frame, (x, height - my), 7, (0, 0, 255), -1)

        # AB and BC lines if available
        if len(points) >= 2:
            cv2.line(display_frame, (points[0][0], height - points[0][1]),
                     (points[1][0], height - points[1][1]), (0, 255, 0), 2)
        if len(points) == 3:
            cv2.line(display_frame, (points[1][0], height - points[1][1]),
                     (points[2][0], height - points[2][1]), (0, 255, 0), 2)

        # overlay if already computed
        if overlay_drawn is not None:
            display_frame = cv2.addWeighted(display_frame, 1.0, overlay_drawn, 0.6, 0)

        cv2.putText(display_frame, f"{image_file} - Click 3 points: 's'=Save, 'q'=Skip, 'r'=Reset",
                    (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        cv2.imshow("Define L-Shape ROI", display_frame)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('r'):
            points = []
            overlay_drawn = None

        elif key == ord('s') and len(points) == 3:
            A, B, C = points
            overlay_drawn, result_data = draw_shaded_region(frame, A, B, C)
            with open(json_path, 'w') as f:
                json.dump(result_data, f, indent=2)
            print(f"[✅] Saved {json_path}")
            break

        elif key == ord('q'):
            print(f"[⚠] Skipped {image_file}")
            break

    cv2.destroyAllWindows()

print("\n✅ Batch L-shape definition complete.")

[✅] Saved output_json/a_lshape.json
[✅] Saved output_json/b_lshape.json
[✅] Saved output_json/c_lshape.json
[✅] Saved output_json/d_lshape.json
[✅] Saved output_json/plain_lshape.json

✅ Batch L-shape definition complete.


: 